# 01. Обзор данных H&M

Знакомимся с тремя исходными таблицами, проверяем ключи, типы, пропуски и диапазон дат. На этом этапе данные не очищаются и модели не обучаются.

## Подключение среды

В Colab проект читается с Google Drive. При локальном запуске используется текущая папка проекта. Все дальнейшие пути строятся от `PROJECT_ROOT`.

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/fashion-recommender-system')
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('Не найдена папка src: ' + str(PROJECT_ROOT / 'src'))
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Корень проекта:', PROJECT_ROOT)

В новой Colab-сессии зависимости устанавливаются из одного файла проекта. Локально этот шаг пропускается, если окружение уже подготовлено.

In [ ]:
import subprocess

if 'google.colab' in sys.modules:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')],
        check=True,
    )

## Загрузка таблиц

Загрузчики проверяют файлы и столбцы, преобразуют дату и централизованно приводят `article_id` к десяти символам.

In [ ]:
import pandas as pd
from IPython.display import display
from fashion_recommender.data import load_articles, load_customers, load_transactions

### Пути к CSV

**Что делаем:** задаём три входных файла.  
**Зачем:** все загрузчики должны читать один каталог.  
**Что получим:** понятные пути к исходным таблицам.

In [ ]:
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
TRANSACTIONS_PATH = RAW_DATA_DIR / 'transactions_train.csv'
ARTICLES_PATH = RAW_DATA_DIR / 'articles.csv'
CUSTOMERS_PATH = RAW_DATA_DIR / 'customers.csv'

### Чтение данных

**Что делаем:** загружаем три таблицы.  
**Зачем:** загрузчики проверят схемы и типы ID.  
**Что получим:** `transactions`, `articles`, `customers`.

In [ ]:
transactions = load_transactions(TRANSACTIONS_PATH)
articles = load_articles(ARTICLES_PATH)
customers = load_customers(CUSTOMERS_PATH)
print('Таблицы загружены')

## Размеры и первые строки

`shape` показывает число строк и столбцов, а `head()` помогает проверить смысл полей без тяжёлых объединений.

In [ ]:
table_sizes = pd.DataFrame({
    'table': ['transactions', 'articles', 'customers'],
    'rows': [len(transactions), len(articles), len(customers)],
    'columns': [transactions.shape[1], articles.shape[1], customers.shape[1]],
})
display(table_sizes)
display(transactions.head())
display(articles.head())
display(customers.head())

## Типы и пропуски

Ключевые ID должны быть строками, а `t_dat` — датой. Процент пропусков удобнее абсолютного числа при сравнении таблиц разного размера.

In [ ]:
for table_name, frame in {'transactions': transactions, 'articles': articles, 'customers': customers}.items():
    print(f'\n{table_name}')
    display(frame.dtypes.to_frame('dtype'))
    display((frame.isna().mean().mul(100).sort_values(ascending=False).to_frame('missing_percent')).head(15))

## Уникальность и полные дубликаты

Повтор одной транзакции может означать покупку нескольких единиц, поэтому строки только считаются, но не удаляются.

In [ ]:
print('Полные дубликаты транзакций:', transactions.duplicated().sum())
print('Повторяющиеся article_id в articles:', articles['article_id'].duplicated().sum())
print('Повторяющиеся customer_id в customers:', customers['customer_id'].duplicated().sum())
print('Уникальные пользователи в покупках:', transactions['customer_id'].nunique())
print('Уникальные товары в покупках:', transactions['article_id'].nunique())

## Временной диапазон и связи

Проверка множеств ключей обнаруживает товары или пользователей, для которых нет описания в справочнике, без создания полного merge.

In [ ]:
unknown_articles = set(transactions['article_id']) - set(articles['article_id'])
unknown_customers = set(transactions['customer_id']) - set(customers['customer_id'])
print('Первая дата:', transactions['t_dat'].min())
print('Последняя дата:', transactions['t_dat'].max())
print('Товаров без описания:', len(unknown_articles))
print('Пользователей без профиля:', len(unknown_customers))

## Итог

Сводка ниже фиксирует фактический объём данных. Нормализованные ключи позволяют безопасно соединять таблицы в следующих этапах.

In [ ]:
summary = {
    'transactions': len(transactions),
    'customers_in_transactions': transactions['customer_id'].nunique(),
    'articles_in_transactions': transactions['article_id'].nunique(),
    'date_min': transactions['t_dat'].min(),
    'date_max': transactions['t_dat'].max(),
}
display(pd.Series(summary, name='value'))